
# diagnose_pyr_layers

Load the 4 trained Pyramid layers (MeasureA/CombineA/MeasureB/CombineB) for a given level d,
run converter-generated inputs at different betas, and print diagnostics tables:

- sign_%: percent of outputs whose sign matches the bit target (logit>=0 <-> bit=1)
- |logit|: mean absolute logit magnitude
- For CombineB gun head: idx_acc_%: percent argmax index correct (one-hot target)

Example:
    python diagnose_pyr_layers.py --d 0 --betas 0.01,0.03,0.05,0.1,0.3,0.5,1,3,5,10
"""

In [5]:
"""
diagnose_pyr_layers.py

Load the 4 trained Pyramid layers (MeasureA/CombineA/MeasureB/CombineB) for a given level d,
run converter-generated inputs at different betas, and print diagnostics tables:

- sign_%: percent of outputs whose sign matches the bit target (logit>=0 <-> bit=1)
- |logit|: mean absolute logit magnitude
- For CombineB gun head: idx_acc_%: percent argmax index correct (one-hot target)

Inspired by `debug_layers_and_models.ipynb`.

Example:
    python diagnose_pyr_layers.py --d 0 --betas 0.01,0.03,0.05,0.1,0.3,0.5,1,3,5,10
"""
from __future__ import annotations

import argparse
import os
import sys
from dataclasses import dataclass
from pathlib import Path
from typing import Literal, Sequence

import numpy as np
import os

# Silence TensorFlow C++ logs (INFO + WARNING)
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"  # 0=all, 1=INFO off, 2=WARNING off, 3=ERROR only

# Optional: reduce Python-side TF logging
import logging
logging.getLogger("tensorflow").setLevel(logging.ERROR)
import tensorflow as tf

LayerKind = Literal["meas_a", "comb_a", "meas_b", "comb_b"]


def change_to_repo_root(marker: str = "WIP") -> None:
    here = Path.cwd()
    for parent in [here] + list(here.parents):
        if (parent / marker).is_dir():
            os.chdir(parent)
            return
    raise RuntimeError(f"Could not find repo root containing '{marker}/' from {here}")


def add_repo_to_syspath() -> None:
    root = Path.cwd()
    if str(root) not in sys.path:
        sys.path.insert(0, str(root))
    wip_src = root / "WIP" / "src"
    core_src = root / "src"
    if wip_src.is_dir() and str(wip_src) not in sys.path:
        sys.path.insert(0, str(wip_src))
    if core_src.is_dir() and str(core_src) not in sys.path:
        sys.path.insert(0, str(core_src))


@dataclass(frozen=True)
class DiagnoseSettings:
    field_size: int = 4
    seed: int = 1234
    num_games: int = 150_000
    num_samples: int = 2000
    hidden_units: int = 64
    d: int = 0
    weights_dir: Path = Path("WIP/weights_pyr_layers")
    filename_template: str = "{kind}_d{d}.weights.h5"


def infer_n2_depth(field_size: int) -> tuple[int, int]:
    n2 = field_size * field_size
    depth = int(np.log2(n2))
    if 2 ** depth != n2:
        raise ValueError(f"field_size^2 must be a power of 2; got n2={n2}.")
    return n2, depth


def level_sizes(n2: int, d: int) -> tuple[int, int]:
    Ld = n2 // (2 ** d)
    kd = Ld // 2
    return Ld, kd


def weights_path(settings: DiagnoseSettings, kind: LayerKind) -> Path:
    return Path(settings.weights_dir) / settings.filename_template.format(kind=kind, d=settings.d)


def wrap_layer_as_model(layer: tf.keras.layers.Layer, kind: LayerKind, *, n2: int, d: int) -> tf.keras.Model:
    Ld, kd = level_sizes(n2, d)
    if kind in ("meas_a", "meas_b"):
        inp = tf.keras.Input(shape=(Ld,), dtype=tf.float32)
        out = layer(inp)
        return tf.keras.Model(inp, out)
    if kind == "comb_a":
        f = tf.keras.Input(shape=(Ld,), dtype=tf.float32)
        o = tf.keras.Input(shape=(kd,), dtype=tf.float32)
        out = layer(f, o)
        return tf.keras.Model([f, o], out)
    if kind == "comb_b":
        g = tf.keras.Input(shape=(Ld,), dtype=tf.float32)
        o = tf.keras.Input(shape=(kd,), dtype=tf.float32)
        c = tf.keras.Input(shape=(1,), dtype=tf.float32)
        out = layer(g, o, c)
        return tf.keras.Model([g, o, c], out)
    raise ValueError(kind)


def load_layer(kind: LayerKind, settings: DiagnoseSettings, *, n2: int) -> tf.keras.layers.Layer:
    from Q_Sea_Battle_New.pyr_measurement_layer_a import PyrMeasurementLayerA
    from Q_Sea_Battle_New.pyr_combine_layer_a import PyrCombineLayerA
    from Q_Sea_Battle_New.pyr_measurement_layer_b import PyrMeasurementLayerB
    from Q_Sea_Battle_New.pyr_combine_layer_b import PyrCombineLayerB

    if kind == "meas_a":
        layer = PyrMeasurementLayerA(hidden_units=settings.hidden_units)
    elif kind == "comb_a":
        layer = PyrCombineLayerA(hidden_units=settings.hidden_units)
    elif kind == "meas_b":
        layer = PyrMeasurementLayerB(hidden_units=settings.hidden_units)
    elif kind == "comb_b":
        layer = PyrCombineLayerB(hidden_units=settings.hidden_units)
    else:
        raise ValueError(kind)

    model = wrap_layer_as_model(layer, kind, n2=n2, d=settings.d)
    # build (Keras 3: avoid tf.* ops on KerasTensor inputs)
    Ld, kd = level_sizes(n2, settings.d)
    if kind in ("meas_a", "meas_b"):
        _ = model(tf.zeros((1, Ld), tf.float32), training=False)
    elif kind == "comb_a":
        _ = model([tf.zeros((1, Ld), tf.float32), tf.zeros((1, kd), tf.float32)], training=False)
    elif kind == "comb_b":
        _ = model([tf.zeros((1, Ld), tf.float32), tf.zeros((1, kd), tf.float32), tf.zeros((1, 1), tf.float32)], training=False)
    else:
        raise ValueError(kind)

    wp = weights_path(settings, kind)
    if not wp.exists():
        raise FileNotFoundError(f"Missing weights for {kind} at: {wp}")
    model.load_weights(str(wp))
    return layer


def sample_idx(N: int, k: int, seed: int) -> np.ndarray:
    rng = np.random.default_rng(seed)
    if k >= N:
        return np.arange(N)
    return rng.choice(N, size=k, replace=False)


def pct(x: float) -> str:
    return f"{100.0 * x:6.2f}"


def mean_abs(x: np.ndarray) -> str:
    return f"{np.mean(np.abs(x)):10.4f}"


def sign_match_pct(logits: np.ndarray, y_bits: np.ndarray) -> float:
    pred_bits = (logits >= 0.0).astype(np.float32)
    return float(np.mean(pred_bits == y_bits))


def print_table(title: str, rows: list[tuple[str, str, str, str | None]], *, include_idx_acc: bool) -> None:
    print()
    print(title)
    if include_idx_acc:
        print(f"{'beta':>6}  {'sign_%':>10}  {'|logit|':>12}  {'idx_acc_%':>10}")
        print(f"{'-'*6}  {'-'*10}  {'-'*12}  {'-'*10}")
        for beta, signp, mag, idxacc in rows:
            print(f"{beta:>6}  {signp:>10}  {mag:>12}  {idxacc:>10}")
    else:
        print(f"{'beta':>6}  {'sign_%':>10}  {'|logit|':>12}")
        print(f"{'-'*6}  {'-'*10}  {'-'*12}")
        for beta, signp, mag, _ in rows:
            print(f"{beta:>6}  {signp:>10}  {mag:>12}")


def run(settings: DiagnoseSettings, betas: Sequence[float]) -> None:
    tf.random.set_seed(settings.seed)
    np.random.seed(settings.seed)

    n2, depth = infer_n2_depth(settings.field_size)
    if not (0 <= settings.d < depth):
        raise ValueError(f"d must be in [0, {depth-1}] for n2={n2}")

    from Q_Sea_Battle_New.pyr_dataset_generation_utilities import generate_pyr_dataset
    from Q_Sea_Battle_New.pyr_dataset_conversion_utilities import (
        convert_layer_measure_a,
        convert_layer_combine_a,
        convert_layer_measure_b,
        convert_layer_combine_b,
    )

    meas_a = load_layer("meas_a", settings, n2=n2)
    comb_a = load_layer("comb_a", settings, n2=n2)
    meas_b = load_layer("meas_b", settings, n2=n2)
    comb_b = load_layer("comb_b", settings, n2=n2)

    ds_np = generate_pyr_dataset(n2=n2, num_games=settings.num_games, seed=settings.seed, validate=True)

    rows_meas_a, rows_comb_a, rows_meas_b, rows_comb_b = [], [], [], []

    for beta in betas:
        # Measure A
        conv = convert_layer_measure_a(ds_np, rep_x="hard_logit", rep_y="bits", beta=[float(beta)])
        X, Y = conv[settings.d]
        idx = sample_idx(len(Y), settings.num_samples, settings.seed + 101)
        logits = meas_a(tf.constant(X[idx], tf.float32), training=False).numpy()
        rows_meas_a.append((f"{beta:g}", pct(sign_match_pct(logits, Y[idx])), mean_abs(logits), None))

        # Combine A
        conv = convert_layer_combine_a(ds_np, rep_field="hard_logit", rep_outcome="hard_logit", rep_target="bits", beta=[float(beta)])
        (field_d, out_a_d), field_d1 = conv[settings.d]
        idx = sample_idx(len(field_d1), settings.num_samples, settings.seed + 202)
        logits = comb_a(tf.constant(field_d[idx], tf.float32), tf.constant(out_a_d[idx], tf.float32), training=False).numpy()
        rows_comb_a.append((f"{beta:g}", pct(sign_match_pct(logits, field_d1[idx])), mean_abs(logits), None))

        # Measure B
        conv = convert_layer_measure_b(ds_np, rep_x="hard_logit", rep_y="bits", beta=[float(beta)])
        X, Y = conv[settings.d]
        idx = sample_idx(len(Y), settings.num_samples, settings.seed + 303)
        logits = meas_b(tf.constant(X[idx], tf.float32), training=False).numpy()
        rows_meas_b.append((f"{beta:g}", pct(sign_match_pct(logits, Y[idx])), mean_abs(logits), None))

        # Combine B (gun + comm)
        conv = convert_layer_combine_b(
            ds_np,
            rep_gun="hard_logit",
            rep_outcome_b="hard_logit",
            rep_comm_in="hard_logit",
            rep_gun_next="bits",
            rep_comm_next="bits",
            beta=[float(beta)],
        )
        (gun_d, out_b_d, comm_d), (gun_d1_onehot, comm_d1_bits) = conv[settings.d]
        idx = sample_idx(len(comm_d1_bits), settings.num_samples, settings.seed + 404)

        gun_logits, comm_logits = comb_b(
            tf.constant(gun_d[idx], tf.float32),
            tf.constant(out_b_d[idx], tf.float32),
            tf.constant(comm_d[idx], tf.float32),
            training=False,
        )
        gun_logits = gun_logits.numpy()
        comm_logits = comm_logits.numpy()

        pred_idx = np.argmax(gun_logits, axis=-1)
        true_idx = np.argmax(gun_d1_onehot[idx], axis=-1)
        idx_acc = float(np.mean(pred_idx == true_idx))

        comm_sign = sign_match_pct(comm_logits, comm_d1_bits[idx])
        rows_comb_b.append((f"{beta:g}", pct(comm_sign), mean_abs(comm_logits), pct(idx_acc)))

    print_table(f"=== Measure A (d={settings.d}) ===", rows_meas_a, include_idx_acc=False)
    print_table(f"=== Combine A (d={settings.d}) ===", rows_comb_a, include_idx_acc=False)
    print_table(f"=== Measure B (d={settings.d}) ===", rows_meas_b, include_idx_acc=False)
    print_table(f"=== Combine B (d={settings.d}) [comm + gun] ===", rows_comb_b, include_idx_acc=True)


def parse_betas(s: str) -> list[float]:
    return [float(x) for x in s.split(",") if x.strip() != ""]



In [6]:
# ===============================
# Notebook version of main()
# ===============================

from pathlib import Path

# ---- Configuration (edit here) ----
FIELD_SIZE = 4
MAX_D = 3
SEED = 1234
NUM_GAMES = 150_000
NUM_SAMPLES = 2000
HIDDEN_UNITS = 64

BETAS_STR = "0.01,0.03,0.05,0.1,0.3,0.5,1,3,5,10"

WEIGHTS_DIR = "WIP/weights_pyr_layers"
FILENAME_TEMPLATE = "{kind}_d{d}.weights.h5"
REPO_ROOT_MARKER = "WIP"

# ---- Optional: change working directory to repo root ----
change_to_repo_root(REPO_ROOT_MARKER)
add_repo_to_syspath()

# ---- Run diagnostics ----
betas = parse_betas(BETAS_STR)

for new_d in range(MAX_D + 1):

    settings = DiagnoseSettings(
        field_size=FIELD_SIZE,
        seed=SEED,
        num_games=NUM_GAMES,
        num_samples=NUM_SAMPLES,
        hidden_units=HIDDEN_UNITS,
        d=new_d,
        weights_dir=Path(WEIGHTS_DIR),
        filename_template=FILENAME_TEMPLATE,
    )

    print(f"\n\n=== Diagnosing Pyramid layers at level d={new_d} ===")
    run(settings, betas)

print("\nDone.")



=== Diagnosing Pyramid layers at level d=0 ===

=== Measure A (d=0) ===
  beta      sign_%       |logit|
------  ----------  ------------
  0.01       52.71        0.1782
  0.03       54.06        0.1775
  0.05       55.24        0.1787
   0.1       69.50        0.1966
   0.3       94.13        0.4064
   0.5      100.00        0.7067
     1      100.00        1.4636
     3      100.00        4.4566
     5      100.00        7.4403
    10      100.00       14.8929

=== Combine A (d=0) ===
  beta      sign_%       |logit|
------  ----------  ------------
  0.01       49.38        0.2300
  0.03       49.70        0.2250
  0.05       57.57        0.2240
   0.1       75.10        0.2650
   0.3      100.00        0.7063
   0.5      100.00        1.2240
     1      100.00        2.5114
     3      100.00        7.6284
     5      100.00       12.7327
    10      100.00       25.4828

=== Measure B (d=0) ===
  beta      sign_%       |logit|
------  ----------  ------------
  0.01       93.67

=== Diagnosing Pyramid layers at level d=0 ===
WARNING:tensorflow:From c:\Users\nly99857\OneDrive - Philips\SW Projects\QSeaBattle\venvs\env_QSeaBattle\Lib\site-packages\keras\src\backend\tensorflow\core.py:232: The name tf.placeholder is deprecated. Please use tf.compat.v1.placeholder instead.


=== Measure A (d=0) ===
  beta      sign_%       |logit|
------  ----------  ------------
  0.01       49.95        0.2087
  0.03       49.95        0.2025
  0.05       50.97        0.1941
   0.1       63.14        0.1919
   0.3       95.36        0.4024
   0.5      100.00        0.7072
     1      100.00        1.4698
     3      100.00        4.4924
     5      100.00        7.5038
    10      100.00       15.0219

=== Combine A (d=0) ===
  beta      sign_%       |logit|
------  ----------  ------------
  0.01       50.79        0.2188
  0.03       53.02        0.2146
  0.05       59.55        0.2119
   0.1       76.25        0.2351
   0.3      100.00        0.7031
   0.5      100.00        1.2179
     1      100.00        2.4978
     3      100.00        7.5917
     5      100.00       12.6712
    10      100.00       25.3555

=== Measure B (d=0) ===
  beta      sign_%       |logit|
------  ----------  ------------
  0.01       93.67        0.2040
  0.03       93.67        0.2380
  0.05       93.67        0.2722
   0.1       95.32        0.3578
   0.3      100.00        0.7276
   0.5      100.00        1.1042
     1      100.00        2.0490
     3      100.00        5.8308
     5      100.00        9.6126
    10      100.00       19.0669

=== Combine B (d=0) [comm + gun] ===
  beta      sign_%       |logit|   idx_acc_%
------  ----------  ------------  ----------
  0.01       52.30        0.0101       13.20
  0.03       58.25        0.0208       13.20
  0.05       63.60        0.0330       13.20
   0.1       77.50        0.0575       26.80
   0.3       99.85        0.1779       38.70
   0.5      100.00        0.3127       62.10
     1      100.00        0.6473      100.00
     3      100.00        1.9810      100.00
     5      100.00        3.3108      100.00
    10      100.00        6.6334      100.00


=== Diagnosing Pyramid layers at level d=1 ===

=== Measure A (d=1) ===
  beta      sign_%       |logit|
------  ----------  ------------
  0.01       49.64        0.1583
  0.03       49.64        0.1630
  0.05       49.64        0.1622
   0.1       68.03        0.1731
   0.3      100.00        0.4222
   0.5      100.00        0.7286
     1      100.00        1.4866
     3      100.00        4.4946
     5      100.00        7.4983
    10      100.00       15.0049

=== Combine A (d=1) ===
  beta      sign_%       |logit|
------  ----------  ------------
  0.01       50.99        0.2756
  0.03       50.99        0.2663
  0.05       57.09        0.2594
   0.1       62.95        0.2725
   0.3      100.00        0.6812
   0.5      100.00        1.2060
     1      100.00        2.5048
     3      100.00        7.6557
     5      100.00       12.7882
    10      100.00       25.6026

=== Measure B (d=1) ===
  beta      sign_%       |logit|
------  ----------  ------------
  0.01       87.49        0.3444
  0.03       87.49        0.3710
  0.05       87.49        0.3975
   0.1       87.49        0.4619
   0.3      100.00        0.7533
   0.5      100.00        1.0990
     1      100.00        1.9614
     3      100.00        5.4148
     5      100.00        8.8680
    10      100.00       17.5005

=== Combine B (d=1) [comm + gun] ===
  beta      sign_%       |logit|   idx_acc_%
------  ----------  ------------  ----------
  0.01       49.45        0.1840       24.85
  0.03       49.45        0.1977       24.85
  0.05       49.45        0.2108       24.85
   0.1       49.45        0.2345       24.85
   0.3       52.70        0.2552       75.95
   0.5       86.75        0.2981      100.00
     1      100.00        0.6234      100.00
     3      100.00        1.9670      100.00
     5      100.00        3.3010      100.00
    10      100.00        6.6256      100.00


=== Diagnosing Pyramid layers at level d=2 ===

=== Measure A (d=2) ===
  beta      sign_%       |logit|
------  ----------  ------------
  0.01       49.53        0.3515
  0.03       49.53        0.3456
  0.05       49.53        0.3384
   0.1       49.53        0.3177
   0.3      100.00        0.4109
   0.5      100.00        0.7338
     1      100.00        1.5338
     3      100.00        4.7061
     5      100.00        7.8711
    10      100.00       15.7815

=== Combine A (d=2) ===
  beta      sign_%       |logit|
------  ----------  ------------
  0.01       62.98        0.1135
  0.03       76.02        0.1266
  0.05       76.02        0.1519
   0.1       88.62        0.2279
   0.3      100.00        0.7410
   0.5      100.00        1.2506
     1      100.00        2.5202
     3      100.00        7.5869
     5      100.00       12.6502
    10      100.00       25.3068

=== Measure B (d=2) ===
  beta      sign_%       |logit|
------  ----------  ------------
  0.01       74.30        0.4605
  0.03       74.30        0.4670
  0.05       74.30        0.4747
   0.1       87.33        0.5045
   0.3      100.00        0.7841
   0.5      100.00        1.1390
     1      100.00        1.9813
     3      100.00        5.3234
     5      100.00        8.6797
    10      100.00       17.0787

=== Combine B (d=2) [comm + gun] ===
  beta      sign_%       |logit|   idx_acc_%
------  ----------  ------------  ----------
  0.01       51.20        0.0303       48.90
  0.03       47.80        0.0214       48.90
  0.05       50.60        0.0202       48.90
   0.1       57.55        0.0346       74.00
   0.3       97.00        0.1280      100.00
   0.5      100.00        0.2670      100.00
     1      100.00        0.6238      100.00
     3      100.00        1.9707      100.00
     5      100.00        3.3055      100.00
    10      100.00        6.6248      100.00


=== Diagnosing Pyramid layers at level d=3 ===

=== Measure A (d=3) ===
  beta      sign_%       |logit|
------  ----------  ------------
  0.01       50.45        0.2733
  0.03       50.45        0.2684
  0.05       50.45        0.2657
   0.1       50.45        0.2586
   0.3       74.55        0.3832
   0.5      100.00        0.6262
     1      100.00        1.5114
     3      100.00        4.9068
     5      100.00        8.2461
    10      100.00       16.5452

=== Combine A (d=3) ===
  beta      sign_%       |logit|
------  ----------  ------------
  0.01       47.10        0.3147
  0.03       47.10        0.3047
  0.05       47.10        0.2902
   0.1       47.10        0.2524
   0.3      100.00        0.5826
   0.5      100.00        1.1118
     1      100.00        2.4285
     3      100.00        7.6161
     5      100.00       12.7838
    10      100.00       25.6742

=== Measure B (d=3) ===
  beta      sign_%       |logit|
------  ----------  ------------
  0.01       49.15        0.0959
  0.03      100.00        0.1045
  0.05      100.00        0.1753
   0.1      100.00        0.3523
   0.3      100.00        0.9555
   0.5      100.00        1.3445
     1      100.00        2.1364
     3      100.00        5.2884
     5      100.00        8.4395
    10      100.00       16.3174

=== Combine B (d=3) [comm + gun] ===
  beta      sign_%       |logit|   idx_acc_%
------  ----------  ------------  ----------
  0.01       51.05        0.0596      100.00
  0.03       51.05        0.0612      100.00
  0.05       51.05        0.0616      100.00
   0.1       76.00        0.0757      100.00
   0.3      100.00        0.1610      100.00
   0.5      100.00        0.2958      100.00
     1      100.00        0.6303      100.00
     3      100.00        1.9694      100.00
     5      100.00        3.3016      100.00
    10      100.00        6.6250      100.00

Done.